Bismillah
Startiong Project on:
Saturday, 26th Rabi-ul-Awwal, 1447 - 20th October, 2025

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier, StackingClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve

from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

import re
import string
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# For advanced preprocessing
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from textblob import TextBlob

import matplotlib.pyplot as plt
import seaborn as sns

# Download NLTK resources
try:
    nltk.download('stopwords', quiet=True)
    nltk.download('wordnet', quiet=True)
    nltk.download('omw-1.4', quiet=True)
    nltk.download('averaged_perceptron_tagger', quiet=True)
except:
    print("Warning: Some NLTK downloads failed. Some features may not work.")


In [ ]:
# ============================================================================
# 1. LOAD YOUR DATA
# ============================================================================
# Replace 'your_dataset.csv' with your actual file path
df = pd.read_csv('Data/combined_dataset.csv')

print("Dataset shape:", df.shape)
print("Class distribution:\n", df['label'].value_counts())
print(f"Class imbalance ratio: {df['label'].value_counts()[0] / df['label'].value_counts()[1]:.2f}:1")
print("\nSample rows:\n", df.head(10))

In [ ]:
# ============================================================================
# 2. FEATURE ENGINEERING
# ============================================================================

def extract_text_features(text):
    """Extract handcrafted features from text"""
    features = {}
    
    # Basic length features
    features['text_length'] = len(text)
    features['word_count'] = len(text.split())
    features['avg_word_length'] = np.mean([len(word) for word in text.split()]) if text.split() else 0
    
    # Special character counts
    features['exclamation_count'] = text.count('!')
    features['question_count'] = text.count('?')
    features['uppercase_count'] = sum(1 for c in text if c.isupper())
    features['uppercase_ratio'] = features['uppercase_count'] / len(text) if len(text) > 0 else 0
    features['special_char_count'] = sum(1 for c in text if c in string.punctuation)
    features['digit_count'] = sum(1 for c in text if c.isdigit())
    
    # Profanity indicators (common toxic words)
    profanity_list = ['hate', 'stupid', 'idiot', 'kill', 'die', 'damn', 'hell', 'shut', 'fuck', 'shit', 'ass', 'bitch']
    features['profanity_count'] = sum(1 for word in text.lower().split() if word in profanity_list)
    
    # Sentiment analysis
    # try:
    #     try:
    #         blob = TextBlob(str(text))  # Ensure the input is a string
    #         sentiment = blob.sentiment
    #         features['sentiment_polarity'] = sentiment.polarity
    #         features['sentiment_subjectivity'] = sentiment.subjectivity
    #     except Exception as e:
    #         print(f"Error processing text: {text}. Error: {e}")
    #         features['sentiment_polarity'] = 0
    #         features['sentiment_subjectivity'] = 0
    # except:
    #     features['sentiment_polarity'] = 0
    #     features['sentiment_subjectivity'] = 0
    
    return features

print("\nExtracting text features...")
text_features = df['text'].apply(extract_text_features).apply(pd.Series)
print("Text features extracted:")
print(text_features.head())

In [ ]:
# ============================================================================
# 3. PREPROCESSING FUNCTIONS
# ============================================================================

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_minimal(text):
    """Minimal preprocessing - just lowercase"""
    return text.lower().strip()

def preprocess_basic(text):
    """Basic preprocessing - lowercase, remove extra spaces"""
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)
    return text

def preprocess_moderate(text):
    """Moderate preprocessing - remove URLs, mentions, hashtags"""
    text = text.lower().strip()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text

def preprocess_aggressive(text):
    """Aggressive preprocessing - remove punctuation, numbers, special chars"""
    text = text.lower().strip()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'[0-9]+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text)
    return text

def preprocess_with_stopwords(text):
    """Preprocessing with stopword removal"""
    text = preprocess_moderate(text)
    words = text.split()
    text = ' '.join([w for w in words if w not in stop_words])
    return text

def preprocess_with_stemming(text):
    """Preprocessing with stemming"""
    text = preprocess_moderate(text)
    words = text.split()
    text = ' '.join([stemmer.stem(w) for w in words])
    return text

def preprocess_with_lemmatization(text):
    """Preprocessing with lemmatization"""
    text = preprocess_moderate(text)
    words = text.split()
    text = ' '.join([lemmatizer.lemmatize(w) for w in words])
    return text

In [ ]:
# ============================================================================
# 4. EXPERIMENT CONFIGURATION
# ============================================================================

preprocessing_techniques = {
    'minimal': preprocess_minimal,
    'basic': preprocess_basic,
    'moderate': preprocess_moderate,
    'aggressive': preprocess_aggressive,
    'with_stopwords': preprocess_with_stopwords,
    'with_stemming': preprocess_with_stemming,
    'with_lemmatization': preprocess_with_lemmatization,
}

vectorization_configs = {
    'tfidf_basic': {
        'type': 'tfidf',
        'params': {'max_features': 5000, 'ngram_range': (1, 1)}
    },
    'tfidf_bigram': {
        'type': 'tfidf',
        'params': {'max_features': 5000, 'ngram_range': (1, 2)}
    },
    'tfidf_trigram': {
        'type': 'tfidf',
        'params': {'max_features': 10000, 'ngram_range': (1, 3)}
    },
    'count_basic': {
        'type': 'count',
        'params': {'max_features': 5000, 'ngram_range': (1, 1)}
    },
    'count_bigram': {
        'type': 'count',
        'params': {'max_features': 5000, 'ngram_range': (1, 2)}
    },
}

sampling_strategies = {
    'none': None,
    'random_oversample': RandomOverSampler(random_state=42),
    'random_undersample': RandomUnderSampler(random_state=42),
    'smote': SMOTE(random_state=42, k_neighbors=3),
}

# Expanded model configurations
model_configs = {
    'logistic_regression_balanced': {
        'model': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
        'name': 'Logistic Regression (Balanced)'
    },
    'logistic_regression_c01': {
        'model': LogisticRegression(max_iter=1000, C=0.1, class_weight='balanced', random_state=42),
        'name': 'Logistic Regression (C=0.1)'
    },
    'logistic_regression_c10': {
        'model': LogisticRegression(max_iter=1000, C=10, class_weight='balanced', random_state=42),
        'name': 'Logistic Regression (C=10)'
    },
    'random_forest_balanced': {
        'model': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1),
        'name': 'Random Forest (100 trees)'
    },
    'random_forest_200': {
        'model': RandomForestClassifier(n_estimators=200, class_weight='balanced', max_depth=20, random_state=42, n_jobs=-1),
        'name': 'Random Forest (200 trees)'
    },
    'naive_bayes': {
        'model': MultinomialNB(),
        'name': 'Naive Bayes'
    },
    'naive_bayes_alpha01': {
        'model': MultinomialNB(alpha=0.1),
        'name': 'Naive Bayes (alpha=0.1)'
    },
    # 'gradient_boosting': {
    #     'model': GradientBoostingClassifier(n_estimators=100, random_state=42),
    #     'name': 'Gradient Boosting'
    # },
    'xgboost': {
        'model': XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss', use_label_encoder=False),
        'name': 'XGBoost'
    },
    'xgboost_tuned': {
        'model': XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, eval_metric='logloss', use_label_encoder=False),
        'name': 'XGBoost (Tuned)'
    },
    'lightgbm': {
        'model': LGBMClassifier(n_estimators=100, random_state=42, verbose=-1),
        'name': 'LightGBM'
    },
    'lightgbm_tuned': {
        'model': LGBMClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, verbose=-1),
        'name': 'LightGBM (Tuned)'
    } #,
    # 'svm_linear': {
    #     'model': SVC(kernel='linear', class_weight='balanced', random_state=42, probability=True),
    #     'name': 'SVM (Linear)'
    # },
}

In [ ]:
# ============================================================================
# 5. CROSS-VALIDATION EXPERIMENT RUNNER
# ============================================================================

results = []
cv_scores_list = []

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], 
    df['label'], 
    test_size=0.2, 
    random_state=42, 
    stratify=df['label']
)

# Also get text features for train/test
text_features_train = text_features.loc[X_train.index]
text_features_test = text_features.loc[X_test.index]

print(f"\nStarting experiments with cross-validation...")
print(f"Train set size: {len(X_train)}, Test set size: {len(X_test)}")
print(f"Train class distribution: {y_train.value_counts().to_dict()}")
print(f"Test class distribution: {y_test.value_counts().to_dict()}\n")

# We'll run a subset of experiments with CV, then all experiments without CV
# To save time, let's do CV on a representative subset
experiments_with_cv = []
experiment_count = 0

# Select a subset for detailed CV analysis (to save time)
cv_experiments = [
    ('moderate', 'tfidf_bigram', 'smote', 'xgboost'),
    ('with_lemmatization', 'tfidf_trigram', 'smote', 'lightgbm'),
    ('moderate', 'tfidf_bigram', 'random_oversample', 'logistic_regression_balanced'),
    ('aggressive', 'count_bigram', 'smote', 'random_forest_balanced'),
]

print("Running detailed cross-validation on selected configurations...\n")

for prep_name, vec_name, samp_name, model_key in cv_experiments:
    prep_func = preprocessing_techniques[prep_name]
    vec_config = vectorization_configs[vec_name]
    sampler = sampling_strategies[samp_name]
    model_config = model_configs[model_key]
    
    print(f"CV: {prep_name} + {vec_name} + {samp_name} + {model_config['name']}")
    
    # Apply preprocessing
    X_train_prep = X_train.apply(prep_func)
    
    # Create vectorizer
    if vec_config['type'] == 'tfidf':
        vectorizer = TfidfVectorizer(**vec_config['params'])
    else:
        vectorizer = CountVectorizer(**vec_config['params'])
    
    # Fit and transform
    X_train_vec = vectorizer.fit_transform(X_train_prep)
    
    # Apply sampling
    if sampler is not None:
        X_train_samp, y_train_samp = sampler.fit_resample(X_train_vec, y_train)
    else:
        X_train_samp, y_train_samp = X_train_vec, y_train
    
    # Cross-validation
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model_config['model'], X_train_samp, y_train_samp, 
                                cv=skf, scoring='f1', n_jobs=-1)
    
    cv_scores_list.append({
        'config': f"{prep_name}+{vec_name}+{samp_name}+{model_config['name']}",
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'cv_scores': cv_scores
    })
    
    print(f"  CV F1 Scores: {cv_scores}")
    print(f"  Mean: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})\n")

In [ ]:
# ============================================================================
# 6. FULL EXPERIMENT RUNNER WITH THRESHOLD TUNING
# ============================================================================

print("\n" + "="*80)
print("Running full experiments on all configurations...")
print("="*80 + "\n")

total_experiments = (len(preprocessing_techniques) * 
                    len(vectorization_configs) * 
                    len(sampling_strategies) * 
                    len(model_configs))

print(f"Total experiments to run: {total_experiments}\n")

# Store models for ensemble later
stored_models = []

for prep_name, prep_func in preprocessing_techniques.items():
    print(f"\n{'='*80}")
    print(f"PREPROCESSING: {prep_name}")
    print(f"{'='*80}")
    
    X_train_prep = X_train.apply(prep_func)
    X_test_prep = X_test.apply(prep_func)
    
    for vec_name, vec_config in vectorization_configs.items():
        print(f"\n  Vectorization: {vec_name}")
        
        if vec_config['type'] == 'tfidf':
            vectorizer = TfidfVectorizer(**vec_config['params'])
        else:
            vectorizer = CountVectorizer(**vec_config['params'])
        
        X_train_vec = vectorizer.fit_transform(X_train_prep)
        X_test_vec = vectorizer.transform(X_test_prep)
        
        for samp_name, sampler in sampling_strategies.items():
            
            if sampler is not None:
                X_train_samp, y_train_samp = sampler.fit_resample(X_train_vec, y_train)
            else:
                X_train_samp, y_train_samp = X_train_vec, y_train
            
            for model_key, model_config in model_configs.items():
                experiment_count += 1
                
                try:
                    model = model_config['model']
                    start_time = datetime.now()
                    
                    model.fit(X_train_samp, y_train_samp)
                    
                    # Default predictions
                    y_pred = model.predict(X_test_vec)
                    
                    # Get probability predictions for threshold tuning
                    if hasattr(model, 'predict_proba'):
                        y_proba = model.predict_proba(X_test_vec)[:, 1]
                        
                        # Find optimal threshold
                        thresholds = np.arange(0.1, 0.9, 0.05)
                        best_threshold = 0.5
                        best_f1_threshold = 0
                        
                        for thresh in thresholds:
                            y_pred_thresh = (y_proba >= thresh).astype(int)
                            f1_thresh = f1_score(y_test, y_pred_thresh)
                            if f1_thresh > best_f1_threshold:
                                best_f1_threshold = f1_thresh
                                best_threshold = thresh
                        
                        # Use optimal threshold
                        y_pred_tuned = (y_proba >= best_threshold).astype(int)
                    else:
                        y_pred_tuned = y_pred
                        best_threshold = 0.5
                        y_proba = None
                    
                    # Calculate metrics (default threshold)
                    accuracy = accuracy_score(y_test, y_pred)
                    f1 = f1_score(y_test, y_pred)
                    precision = precision_score(y_test, y_pred)
                    recall = recall_score(y_test, y_pred)
                    f1_toxic = f1_score(y_test, y_pred, pos_label=1)
                    f1_non_toxic = f1_score(y_test, y_pred, pos_label=0)
                    
                    # Calculate metrics (tuned threshold)
                    accuracy_tuned = accuracy_score(y_test, y_pred_tuned)
                    f1_tuned = f1_score(y_test, y_pred_tuned)
                    precision_tuned = precision_score(y_test, y_pred_tuned)
                    recall_tuned = recall_score(y_test, y_pred_tuned)
                    
                    end_time = datetime.now()
                    duration = (end_time - start_time).total_seconds()
                    
                    # Store results
                    result = {
                        'experiment_id': experiment_count,
                        'preprocessing': prep_name,
                        'vectorization': vec_name,
                        'sampling': samp_name,
                        'model': model_config['name'],
                        'accuracy': accuracy,
                        'f1_score': f1,
                        'precision': precision,
                        'recall': recall,
                        'f1_toxic': f1_toxic,
                        'f1_non_toxic': f1_non_toxic,
                        'best_threshold': best_threshold,
                        'accuracy_tuned': accuracy_tuned,
                        'f1_tuned': f1_tuned,
                        'precision_tuned': precision_tuned,
                        'recall_tuned': recall_tuned,
                        'duration_seconds': duration,
                        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                    }
                    results.append(result)
                    
                    # Store top models for ensemble
                    if f1_tuned > 0.7 and len(stored_models) < 20:
                        stored_models.append({
                            'model': model,
                            'vectorizer': vectorizer,
                            'prep_func': prep_func,
                            'f1': f1_tuned,
                            'config': result
                        })
                    
                    if experiment_count % 20 == 0:
                        print(f"    [{experiment_count}/{total_experiments}] Completed")
                
                except Exception as e:
                    print(f"    ERROR with {model_config['name']}: {str(e)}")
                    results.append({
                        'experiment_id': experiment_count,
                        'preprocessing': prep_name,
                        'vectorization': vec_name,
                        'sampling': samp_name,
                        'model': model_config['name'],
                        'error': str(e)
                    })

print(f"\n{'='*80}")
print("ALL EXPERIMENTS COMPLETED!")
print(f"{'='*80}\n")

In [ ]:
# ============================================================================
# 7. ENSEMBLE MODELS (Voting & Stacking)
# ============================================================================

print("\n" + "="*80)
print("BUILDING ENSEMBLE MODELS FROM TOP PERFORMERS")
print("="*80 + "\n")

# Sort stored models by F1 score and take top 5
stored_models_sorted = sorted(stored_models, key=lambda x: x['f1'], reverse=True)[:5]

if len(stored_models_sorted) >= 3:
    print(f"Building ensemble from top {len(stored_models_sorted)} models...")
    
    # Prepare a common preprocessing and vectorization for ensemble
    # Using moderate preprocessing and tfidf_bigram
    X_train_ensemble = X_train.apply(preprocess_moderate)
    X_test_ensemble = X_test.apply(preprocess_moderate)
    
    vectorizer_ensemble = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
    X_train_vec_ens = vectorizer_ensemble.fit_transform(X_train_ensemble)
    X_test_vec_ens = vectorizer_ensemble.transform(X_test_ensemble)
    
    # Apply SMOTE
    smote_ens = SMOTE(random_state=42, k_neighbors=3)
    X_train_ens_samp, y_train_ens_samp = smote_ens.fit_resample(X_train_vec_ens, y_train) # type: ignore
    
    # Build Voting Classifier
    try:
        estimators = [
            ('lr', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)),
            ('rf', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)),
            ('xgb', XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss', use_label_encoder=False)),
        ]
        
        voting_clf = VotingClassifier(estimators=estimators, voting='soft')
        voting_clf.fit(X_train_ens_samp, y_train_ens_samp)
        
        y_pred_voting = voting_clf.predict(X_test_vec_ens)
        
        results.append({
            'experiment_id': len(results) + 1,
            'preprocessing': 'moderate',
            'vectorization': 'tfidf_bigram',
            'sampling': 'smote',
            'model': 'Voting Ensemble (LR+RF+XGB)',
            'accuracy': accuracy_score(y_test, y_pred_voting),
            'f1_score': f1_score(y_test, y_pred_voting),
            'precision': precision_score(y_test, y_pred_voting),
            'recall': recall_score(y_test, y_pred_voting),
            'f1_toxic': f1_score(y_test, y_pred_voting, pos_label=1),
            'f1_non_toxic': f1_score(y_test, y_pred_voting, pos_label=0),
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        })
        print("✓ Voting Ensemble completed")
    except Exception as e:
        print(f"✗ Voting Ensemble failed: {e}")
    
    # Build Stacking Classifier
    try:
        base_estimators = [
            ('lr', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)),
            ('rf', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)),
            ('xgb', XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss', use_label_encoder=False)),
        ]
        
        stacking_clf = StackingClassifier(
            estimators=base_estimators,
            final_estimator=LogisticRegression(max_iter=1000, random_state=42),
            cv=5
        )
        
        stacking_clf.fit(X_train_ens_samp, y_train_ens_samp)
        y_pred_stacking = stacking_clf.predict(X_test_vec_ens)
        
        results.append({
            'experiment_id': len(results) + 1,
            'preprocessing': 'moderate',
            'vectorization': 'tfidf_bigram',
            'sampling': 'smote',
            'model': 'Stacking Ensemble (LR+RF+XGB)',
            'accuracy': accuracy_score(y_test, y_pred_stacking),
            'f1_score': f1_score(y_test, y_pred_stacking),
            'precision': precision_score(y_test, y_pred_stacking),
            'recall': recall_score(y_test, y_pred_stacking),
            'f1_toxic': f1_score(y_test, y_pred_stacking, pos_label=1),
            'f1_non_toxic': f1_score(y_test, y_pred_stacking, pos_label=0),
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        })
        print("✓ Stacking Ensemble completed")
    except Exception as e:
        print(f"✗ Stacking Ensemble failed: {e}")

In [ ]:
# ============================================================================
# 8. RESULTS ANALYSIS
# ============================================================================

results_df = pd.DataFrame(results)
results_df.to_csv('toxicity_experiment_results.csv', index=False)
print("\n✓ Results saved to 'toxicity_experiment_results.csv'")

# Display top performers by F1 score
print("\n" + "="*80)
print("TOP 15 MODELS BY F1 SCORE (TUNED THRESHOLD)")
print("="*80)
top_f1 = results_df.nlargest(15, 'f1_tuned')[['model', 'preprocessing', 'vectorization', 
                                                'sampling', 'f1_tuned', 'accuracy_tuned', 
                                                'precision_tuned', 'recall_tuned', 'best_threshold']]
print(top_f1.to_string(index=False))

# Best balanced model
print("\n" + "="*80)
print("MOST BALANCED MODELS (Best F1 for toxic class)")
print("="*80)
top_balanced = results_df.nlargest(10, 'f1_toxic')[['model', 'preprocessing', 'vectorization', 
                                                     'sampling', 'f1_toxic', 'f1_non_toxic', 
                                                     'f1_score', 'accuracy']]
print(top_balanced.to_string(index=False))

# Summary statistics
print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)
print(f"Best F1 Score (default): {results_df['f1_score'].max():.4f}")
print(f"Best F1 Score (tuned): {results_df['f1_tuned'].max():.4f}")
print(f"Best Accuracy: {results_df['accuracy'].max():.4f}")
print(f"Average F1 Score: {results_df['f1_score'].mean():.4f}")
print(f"Average Accuracy: {results_df['accuracy'].mean():.4f}")

# Performance summaries
print("\n" + "="*80)
print("AVERAGE PERFORMANCE BY PREPROCESSING TECHNIQUE")
print("="*80)
prep_summary = results_df.groupby('preprocessing')[['accuracy', 'f1_score', 'f1_tuned']].mean().sort_values('f1_tuned', ascending=False)
print(prep_summary)

print("\n" + "="*80)
print("AVERAGE PERFORMANCE BY MODEL TYPE")
print("="*80)
model_summary = results_df.groupby('model')[['accuracy', 'f1_score', 'f1_tuned']].mean().sort_values('f1_tuned', ascending=False)
print(model_summary)

print("\n" + "="*80)
print("AVERAGE PERFORMANCE BY SAMPLING STRATEGY")
print("="*80)
samp_summary = results_df.groupby('sampling')[['accuracy', 'f1_score', 'f1_tuned']].mean().sort_values('f1_tuned', ascending=False)
print(samp_summary)

In [ ]:
# # Get the best model
# best_model_row = results_df.loc[results_df['f1_tuned'].idxmax()]

# # Recreate the best model
# best_model_name = best_model_row['model']
# best_model = None
# for key, config in model_configs.items():
#     if config['name'] == best_model_name:
#         best_model = config['model']
#         break

# if best_model is None:
#     # It might be an ensemble
#     if 'Voting' in best_model_name:
#         estimators = [
#             ('lr', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)),
#             ('rf', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)),
#             ('xgb', XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss', use_label_encoder=False)),
#         ]
#         best_model = VotingClassifier(estimators=estimators, voting='soft')
#     elif 'Stacking' in best_model_name:
#         base_estimators = [
#             ('lr', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)),
#             ('rf', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)),
#             ('xgb', XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss', use_label_encoder=False)),
#         ]
#         best_model = StackingClassifier(estimators=base_estimators, final_estimator=LogisticRegression(max_iter=1000, random_state=42), cv=5)

In [ ]:
# # ============================================================================
# # 10. VISUALIZATIONS
# # ============================================================================

# print("\n" + "="*80)
# print("GENERATING VISUALIZATIONS")
# print("="*80 + "\n")

# # Create a figure with multiple subplots
# fig = plt.figure(figsize=(20, 12))

# # 1. Confusion Matrix
# if best_model is not None:
#     ax1 = plt.subplot(2, 3, 1)
#     cm = confusion_matrix(y_test, y_pred_best)
#     sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1)
#     ax1.set_title(f'Confusion Matrix - Best Model\n{best_model_name}', fontsize=12, fontweight='bold')
#     ax1.set_ylabel('True Label')
#     ax1.set_xlabel('Predicted Label')

# # 2. F1 Score by Preprocessing
# ax2 = plt.subplot(2, 3, 2)
# prep_summary_plot = results_df.groupby('preprocessing')['f1_tuned'].mean().sort_values(ascending=True)
# prep_summary_plot.plot(kind='barh', ax=ax2, color='skyblue')
# ax2.set_title('Average F1 Score by Preprocessing', fontsize=12, fontweight='bold')
# ax2.set_xlabel('F1 Score')
# ax2.set_ylabel('Preprocessing Technique')

# # 3. F1 Score by Model Type
# ax3 = plt.subplot(2, 3, 3)
# model_summary_plot = results_df.groupby('model')['f1_tuned'].mean().sort_values(ascending=False).head(10)
# model_summary_plot.plot(kind='bar', ax=ax3, color='coral')
# ax3.set_title('Top 10 Models by Average F1 Score', fontsize=12, fontweight='bold')
# ax3.set_xlabel('Model')
# ax3.set_ylabel('F1 Score')
# ax3.tick_params(axis='x', rotation=45)

# # 4. F1 Score by Sampling Strategy
# ax4 = plt.subplot(2, 3, 4)
# samp_summary_plot = results_df.groupby('sampling')['f1_tuned'].mean().sort_values(ascending=True)
# samp_summary_plot.plot(kind='barh', ax=ax4, color='lightgreen')
# ax4.set_title('Average F1 Score by Sampling Strategy', fontsize=12, fontweight='bold')
# ax4.set_xlabel('F1 Score')
# ax4.set_ylabel('Sampling Strategy')

# # 5. ROC Curve for best model
# if best_model is not None and hasattr(best_model, 'predict_proba'):
#     ax5 = plt.subplot(2, 3, 5)
#     y_proba_best = best_model.predict_proba(X_test_vec)[:, 1]
#     fpr, tpr, _ = roc_curve(y_test, y_proba_best)
#     roc_auc = auc(fpr, tpr)
    
#     ax5.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
#     ax5.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
#     ax5.set_xlim([0.0, 1.0])
#     ax5.set_ylim([0.0, 1.05])
#     ax5.set_xlabel('False Positive Rate')
#     ax5.set_ylabel('True Positive Rate')
#     ax5.set_title('ROC Curve - Best Model', fontsize=12, fontweight='bold')
#     ax5.legend(loc="lower right")
#     ax5.grid(True, alpha=0.3)

# # 6. Precision-Recall Curve
# if best_model is not None and hasattr(best_model, 'predict_proba'):
#     ax6 = plt.subplot(2, 3, 6)
#     precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_proba_best)
#     pr_auc = auc(recall_curve, precision_curve)
    
#     ax6.plot(recall_curve, precision_curve, color='green', lw=2, label=f'PR curve (AUC = {pr_auc:.2f})')
#     ax6.set_xlim([0.0, 1.0])
#     ax6.set_ylim([0.0, 1.05])
#     ax6.set_xlabel('Recall')
#     ax6.set_ylabel('Precision')
#     ax6.set_title('Precision-Recall Curve - Best Model', fontsize=12, fontweight='bold')
#     ax6.legend(loc="lower left")
#     ax6.grid(True, alpha=0.3)

# plt.tight_layout()
# plt.savefig('toxicity_experiment_visualizations.png', dpi=300, bbox_inches='tight')
# print("✓ Visualizations saved to 'toxicity_experiment_visualizations.png'")

# # Additional visualization: Top 20 models comparison
# fig2, ax = plt.subplots(figsize=(14, 8))
# top_20 = results_df.nlargest(20, 'f1_tuned')[['model', 'f1_tuned', 'accuracy_tuned']].reset_index(drop=True)
# x = np.arange(len(top_20))
# width = 0.35

# bars1 = ax.bar(x - width/2, top_20['f1_tuned'], width, label='F1 Score', color='steelblue')
# bars2 = ax.bar(x + width/2, top_20['accuracy_tuned'], width, label='Accuracy', color='lightcoral')

# ax.set_xlabel('Model Configuration', fontsize=12)
# ax.set_ylabel('Score', fontsize=12)
# ax.set_title('Top 20 Model Configurations: F1 Score vs Accuracy', fontsize=14, fontweight='bold')
# ax.set_xticks(x)
# ax.set_xticklabels([f"{i+1}" for i in range(len(top_20))], rotation=0)
# ax.legend()
# ax.grid(True, axis='y', alpha=0.3)

# # Add value labels on bars
# for bars in [bars1, bars2]:
#     for bar in bars:
#         height = bar.get_height()
#         ax.text(bar.get_x() + bar.get_width()/2., height,
#                 f'{height:.3f}',
#                 ha='center', va='bottom', fontsize=7)

# plt.tight_layout()
# plt.savefig('top_20_models_comparison.png', dpi=300, bbox_inches='tight')
# print("✓ Top 20 models comparison saved to 'top_20_models_comparison.png'")

# # Feature importance visualization (if best model is tree-based)
# if best_model is not None and hasattr(best_model, 'feature_importances_'):
#     fig3, ax = plt.subplots(figsize=(12, 8))
    
#     feature_names = vectorizer.get_feature_names_out()
#     importances = best_model.feature_importances_
#     indices = np.argsort(importances)[-20:]  # Top 20 features
    
#     ax.barh(range(len(indices)), importances[indices], color='teal')
#     ax.set_yticks(range(len(indices)))
#     ax.set_yticklabels([feature_names[i] for i in indices])
#     ax.set_xlabel('Feature Importance')
#     ax.set_title('Top 20 Most Important Features - Best Model', fontsize=14, fontweight='bold')
#     ax.grid(True, axis='x', alpha=0.3)
    
#     plt.tight_layout()
#     plt.savefig('feature_importance.png', dpi=300, bbox_inches='tight')
#     print("✓ Feature importance saved to 'feature_importance.png'")